# Discrete CPS — scikit-learn and LightGBM

CPS for count data, including CDF, PMF, PPF, coverage, and optimal inventory.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyconformal.distribution import DiscreteCrossConformalPredictiveSystem
from tinyconformal.utils import NewsvendorSolver

rng = np.random.default_rng(42)
X = rng.uniform(0, 3, size=(3000, 1))
mu = np.exp(0.7 + 0.55 * X[:, 0])
y = rng.poisson(mu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [2]:
models = {
    "HistGradientBoosting": HistGradientBoostingRegressor(loss="poisson", max_iter=250, random_state=42),
    "LightGBM": LGBMRegressor(objective="poisson", n_estimators=250, random_state=42, verbosity=-1),
}
results = {}
for name, model in models.items():
    scale_model = RandomForestRegressor(n_estimators=250, min_samples_leaf=8, random_state=42, n_jobs=-1)
    cps = DiscreteCrossConformalPredictiveSystem(model, scale_model, cv=5, n_jobs=-1, minimum=0).fit(X_train, y_train)
    distribution = cps.predict_distribution(X_test)
    results[name] = (cps, distribution)
    display(distribution.evaluate(y_test))

,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.625000,3.028333,5.808333
1,0.80,0.865000,5.828333,8.045000
2,0.90,0.945000,7.611667,9.278333
3,0.95,0.966667,8.643333,10.376667


,coverage,empirical_coverage,mean_width,winkler_score
0,0.50,0.638333,3.081667,5.655
1,0.80,0.885000,5.868333,7.785
2,0.90,0.946667,7.536667,9.170
3,0.95,0.968333,8.995000,10.795


## CDF, PMF, PPF, and integer quantiles

In [3]:
distribution = results["HistGradientBoosting"][1]
levels = np.array([0.1, 0.5, 0.9])
level_matrix = np.broadcast_to(levels, (len(distribution), len(levels)))
quantile_predictions = distribution.ppf(level_matrix)

pd.DataFrame({
    "y": y_test[:10],
    "cdf_at_y": distribution.cdf(y_test)[:10],
    "pmf_at_y": distribution.pmf(y_test)[:10],
    "q10": quantile_predictions[:10, 0],
    "q50": quantile_predictions[:10, 1],
    "q90": quantile_predictions[:10, 2],
})

,y,cdf_at_y,pmf_at_y,q10,q50,q90
0,5,0.036606,0.021631,7,9,13
1,5,0.336106,0.148087,4,6,10
2,8,0.422629,0.154742,6,9,12
3,4,0.637271,0.168053,1,4,7
4,7,0.745424,0.171381,3,6,9
5,6,0.871880,0.086522,1,4,7
6,7,0.745424,0.171381,3,6,9
7,6,0.562396,0.174709,3,6,9
8,5,0.797005,0.124792,1,4,7
9,6,0.049917,0.034942,7,10,13


## Inventory solver and marginal benefit

In [4]:
decision_frame = pd.DataFrame({
    "unique_id": np.arange(len(y_test)).astype(str),
    "ds": pd.Timestamp("2026-01-01"),
    "shortage_cost": 9.0,
    "holding_cost": 1.0,
})
stock = NewsvendorSolver.optimize_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
)
assert np.all(stock["y_optimal"] == np.floor(stock["y_optimal"]))
display(stock.head())
marginal_benefit = NewsvendorSolver.marginal_benefit_distribution(
    decision_frame,
    distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    units=[0, 5, 10, 15],
)
marginal_benefit[["unique_id", "MB(k=0)", "MB(k=5)", "MB(k=10)", "MB(k=15)"]].head()

,unique_id,ds,shortage_cost,holding_cost,critical_ratio,y_optimal
0,0,2026-01-01,9.0,1.0,0.9,13.0
1,1,2026-01-01,9.0,1.0,0.9,10.0
2,2,2026-01-01,9.0,1.0,0.9,12.0
3,3,2026-01-01,9.0,1.0,0.9,7.0
4,4,2026-01-01,9.0,1.0,0.9,9.0


,unique_id,MB(k=0),MB(k=5),MB(k=10),MB(k=15)
0,0,9.0,8.850250,3.841930,-0.750416
1,1,9.0,7.119800,0.081531,-0.966722
2,2,9.0,8.783694,3.093178,-0.783694
3,3,9.0,2.627288,-0.800333,-0.966722
4,4,9.0,6.470882,-0.184692,-0.966722
